# 大量の口コミから観点ごとの賛否をまとめる（意見の要約）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_summarization/opinion_summarization.ipynb)

NLP クックブック 27-08 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

1 つのホテルについての数百件の口コミから、「立地は 9 割が高評価、部屋の狭さへの不満が多い」のように、観点ごとの賛否の割合と代表的な意見をまとめた要約を作りたい。要約が多数派の意見を正しく伝えているか、少数の強い苦情を誇張していないかも確かめたい。

In [1]:
import numpy as np
import pandas as pd

# 架空のビジネスホテルの口コミ（このレシピのために作ったもの）。観点ごとに言及する確率と肯定の確率を決めて文を組み合わせるので、正解の割合がわかる
ASPECTS = {   # 値は、言及する確率と、言及したときに肯定である確率
    "立地": (0.5, 0.9), "部屋の広さ": (0.4, 0.2), "清潔さ": (0.35, 0.93), "朝食": (0.4, 0.7), "接客": (0.35, 0.55), "価格": (0.3, 0.75)}
PHRASES = {
    ("立地", "肯定"): ["駅から歩いて 3 分ほどで、とても便利でした。", "駅前なので、雨の日でも移動が楽でした。", "近くにコンビニや飲食店が多く、立地は申し分ありません。",
                      "空港バスの乗り場が目の前で助かりました。", "観光の拠点にちょうどよい場所です。", "繁華街まで歩いて行けるのがよかったです。"],
    ("立地", "否定"): ["駅から少し歩くうえ、坂道がきつかったです。", "周りに食事ができる店が少なく、不便でした。", "入口がわかりにくく、地図を見ても迷いました。"],
    ("部屋の広さ", "肯定"): ["部屋は思ったより広く、スーツケースも広げられました。", "部屋がゆったりしていて、くつろげました。", "ビジネスホテルにしては部屋が広めでした。"],
    ("部屋の広さ", "否定"): ["部屋が狭く、スーツケースを広げる場所がありませんでした。", "ベッドと机だけでいっぱいの、かなり狭い部屋でした。", "二人で泊まるには部屋が窮屈でした。",
                          "部屋は狭めで、荷物の置き場に困りました。", "写真で見るより部屋が小さく感じました。", "部屋が狭いうえに窓も小さく、圧迫感がありました。"],
    ("清潔さ", "肯定"): ["部屋は清潔で、掃除が行き届いていました。", "水回りもきれいで、気持ちよく使えました。", "シーツもタオルも清潔でした。",
                        "古い建物ですが、清掃はしっかりしていました。", "ほこり一つなく、きれいな部屋でした。"],
    ("清潔さ", "否定"): ["夜中に部屋でゴキブリが出ました。本当に最悪で、二度と泊まりません。", "浴室にカビがあり、とても不潔でした。信じられません。",
                        "ベッドに前の客の髪の毛が残っていて、ぞっとしました。"],   # 少数だが強い言葉の苦情
    ("朝食", "肯定"): ["朝食のバイキングは品数が多く、満足しました。", "朝食の焼きたてのパンがおいしかったです。", "地元の食材を使った朝食がよかったです。",
                      "朝食は無料なのに品数が多く、得をした気分でした。"],
    ("朝食", "否定"): ["朝食は品数が少なく、少し残念でした。", "朝食会場が混んでいて、30 分ほど待たされました。", "朝食の料理が冷めていました。"],
    ("接客", "肯定"): ["フロントの方の対応がとても丁寧でした。", "チェックインのとき、スタッフが笑顔で迎えてくれました。", "荷物を早めに預かってもらえて助かりました。",
                      "周辺のおすすめの店を親切に教えてくれました。"],
    ("接客", "否定"): ["フロントの対応がそっけなく、感じがよくありませんでした。", "チェックインに 20 分以上かかり、説明も不十分でした。",
                      "電話で問い合わせたら、たらい回しにされました。", "スタッフの言葉づかいが少し気になりました。"],
    ("価格", "肯定"): ["料金が手ごろで、コストパフォーマンスがよいです。", "平日は安く泊まれるので、出張で毎回使っています。", "値段のわりに設備が整っていました。"],
    ("価格", "否定"): ["週末は料金が高く、割高に感じました。", "駐車場が有料で、思ったより高くつきました。", "この内容でこの料金は、少し高いと思います。"],
}
OPENERS = ["出張で 1 泊しました。", "家族旅行で利用しました。", "友人と 2 泊しました。", "", "", ""]

def make_reviews(n, seed=0):
    rng = np.random.default_rng(seed)
    reviews, gold = [], []
    while len(reviews) < n:
        labels = {a: ("肯定" if rng.random() < p_pos else "否定") for a, (p_mention, p_pos) in ASPECTS.items() if rng.random() < p_mention}
        if not labels:
            continue
        body = "".join(rng.choice(PHRASES[(a, labels[a])]) for a in rng.permutation(list(labels)))
        pos = sum(v == "肯定" for v in labels.values())
        closer = "また利用したいです。" if pos > len(labels) / 2 else "次は別のホテルにします。" if pos < len(labels) / 2 else ""
        reviews.append(rng.choice(OPENERS) + body + closer)
        gold.append(labels)
    return reviews, gold

reviews, gold = make_reviews(200)
gold_df = pd.DataFrame([{"review": i, "aspect": a, "polarity": p} for i, g in enumerate(gold) for a, p in g.items()])
truth = gold_df.groupby("aspect")["polarity"].value_counts().unstack(fill_value=0).reindex(list(ASPECTS))
truth["肯定率"] = (truth["肯定"] / (truth["肯定"] + truth["否定"])).round(3)
print(len(reviews), "件、", sum(map(len, reviews)), "字")
print(truth)
for r in reviews[:3]:
    print("-", r)

200 件、 13508 字
polarity  否定  肯定    肯定率
aspect                 
立地        11  90  0.891
部屋の広さ     70  26  0.271
清潔さ        7  60  0.896
朝食        26  60  0.698
接客        36  36  0.500
価格        21  43  0.672
- 部屋がゆったりしていて、くつろげました。シーツもタオルも清潔でした。また利用したいです。
- 朝食は品数が少なく、少し残念でした。料金が手ごろで、コストパフォーマンスがよいです。古い建物ですが、清掃はしっかりしていました。部屋は狭めで、荷物の置き場に困りました。
- 友人と 2 泊しました。写真で見るより部屋が小さく感じました。次は別のホテルにします。


In [2]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # この章のほかのレシピと同じ 3B の日本語モデル（MIT ライセンス）。初回はダウンロード（約 6.4GB）を含めて数分
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
tokenizer.padding_side = "left"      # 複数の入力をまとめて生成するときは左に詰める
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat_batch(messages_list, max_new_tokens=400):
    texts = [tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages_list]
    inputs = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

chat = lambda prompt, max_new_tokens=400: chat_batch([[{"role": "user", "content": prompt}]], max_new_tokens)[0]
print("文脈長の上限:", llm.config.max_position_embeddings, "トークン")

文脈長の上限: 8192 トークン


In [3]:
# 方法 1: 全部の口コミを 1 回で LLM に渡し、割合と代表的な意見を書かせる。指示は長い入力の後ろに置く
FORMAT = "\n".join(f"- {a}: 肯定 ○%、否定 ○%。代表的な意見" for a in ASPECTS) + "\nまとめ: 全体の評価を 1〜2 文で"
def one_shot_prompt(revs):
    joined = "\n".join(f"[{i}] {r}" for i, r in enumerate(revs, 1))
    return (f"次は、あるビジネスホテルの口コミ {len(revs)} 件です。\n\n{joined}\n\n上の口コミ {len(revs)} 件を要約してください。"
            f"観点ごとに、その観点に触れた口コミのうち肯定的なものと否定的なものの割合と、代表的な意見を書いてください。次の形式で、○ には数字を入れてください。\n{FORMAT}")

for n in [200, 300]:
    print(f"口コミ {n} 件のプロンプト:", len(tokenizer(one_shot_prompt(make_reviews(n)[0]))["input_ids"]), "トークン")

# 口コミの並び順だけを変えて 3 回作る。T4 で 1 回 30 秒〜1 分ほど
one_shot = {}
for seed in [0, 1, 2]:
    order = np.random.default_rng(seed).permutation(len(reviews)) if seed else np.arange(len(reviews))
    one_shot[f"一括（順序 {seed}）"] = out = chat(one_shot_prompt([reviews[i] for i in order]))
    print(f"--- 一括（順序 {seed}）\n{out}")

口コミ 200 件のプロンプト: 7045 トークン
口コミ 300 件のプロンプト: 10623 トークン
--- 一括（順序 0）
立地に関しては、約 60% の口コミが肯定的な意見を寄せており、特に駅に近いことや空港バスの乗り場が近いことが評価されています。一方、約 40% の口コミでは、部屋の広さや清潔さに関する不満が述べられています。部屋の清潔さや広さについては、肯定的な意見が約 30%、否定的な意見が約 70% となっています。朝食については、約 50% の口コミが肯定的な感想を述べていますが、一部ではメニューが物足りないとの意見もあります。接客に関しては、約 70% の口コミで肯定的な対応が評価されていますが、約 30% では対応が不十分だったとの感想があります。価格については、コストパフォーマンスが良いとする意見が約 40% ありますが、価格に対してサービスが見合わないとする意見も約 40% あります。総じて、立地と接客は好評である一方、部屋の広さや清潔さ、朝食の内容や価格については改善の余地があると考えられます。
--- 一括（順序 1）
立地に関しては、約 60% の口コミが肯定的な意見を寄せており、特に「駅前で便利」というコメントが多かった。一方、約 40% の口コミでは部屋が狭く、快適さに欠けるという意見が寄せられた。清潔さについては、約 80% の口コミで肯定的な評価がなされており、清掃状態の良さが高く評価されていた。朝食に関しては、約 70% の口コミで満足の声が聞かれたが、一部で料理の種類が少ないという指摘もあった。接客については、約 50% の口コミで好意的な意見が述べられており、丁寧な対応が評価されていた。しかし、約 50% の口コミでは接客態度や対応に不満を持つ声も聞かれた。価格に関しては、約 30% の口コミでコストパフォーマンスが良いと評価されていたが、約 70% の口コミでは価格が高すぎるという否定的な意見が多かった。総じて、清潔さと立地、朝食には満足している利用者が多いものの、価格と部屋の広さに関しては不満を持つ声が多かった。
--- 一括（順序 2）
立地に関しては、約 60% の口コミが肯定的な意見を寄せており、特に「駅前で便利」といった声が多かった。一方、約 40% の口コミでは部屋が狭く、荷物を置くスペースに困ると

In [4]:
import json
import re

# 方法 2 の段階 1: 口コミを 1 件ずつ LLM に読ませ、観点・評価・根拠の文を JSON で抜き出す
INSTRUCTION = (f"次のホテルの口コミが、観点（{'、'.join(ASPECTS)}）のどれについて、肯定と否定のどちらの評価をしているかを JSON のリストで答えてください。"
               "触れていない観点は含めないでください。根拠には口コミの文をそのまま抜き出してください。JSON 以外は出力しないでください。")
EXAMPLE_IN = "駅から近くて便利でした。ただ、部屋は清掃が行き届いていませんでした。朝食はまあまあです。"
EXAMPLE_OUT = json.dumps([{"観点": "立地", "評価": "肯定", "根拠": "駅から近くて便利でした。"},
                          {"観点": "清潔さ", "評価": "否定", "根拠": "部屋は清掃が行き届いていませんでした。"}], ensure_ascii=False)

def messages(review):
    return [{"role": "user", "content": f"{INSTRUCTION}\n\n口コミ: {EXAMPLE_IN}"}, {"role": "assistant", "content": EXAMPLE_OUT},
            {"role": "user", "content": f"{INSTRUCTION}\n\n口コミ: {review}"}]

def validate(output, review):
    """観点と評価が決められた値の項目だけを残す。根拠が口コミにそのまま出てくるかを ok に記録する。JSON が読めなければ None"""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", output.strip(), flags=re.MULTILINE)
    try:
        items, _ = json.JSONDecoder().raw_decode(text[text.index("["):])
    except ValueError:
        return None
    items = [x for x in items if isinstance(x, dict) and x.get("観点") in ASPECTS and x.get("評価") in ("肯定", "否定")]
    for x in items:
        ev = str(x.get("根拠", "")).strip().rstrip("。．.")
        x["ok"] = len(ev) >= 5 and ev in review
    return items

# 16 件ずつまとめて生成する。T4 で 200 件に 3〜5 分ほど
outputs = []
for i in range(0, len(reviews), 16):
    outputs += chat_batch([messages(r) for r in reviews[i:i + 16]], max_new_tokens=300)
extracted = [validate(o, r) for o, r in zip(outputs, reviews)]
failed = [i for i, e in enumerate(extracted) if e is None]
print("JSON が読めなかった口コミ:", len(failed), "例:", [outputs[i][:120] for i in failed[:2]])
# 読めなかったものは、誤りを伝えて 1 回だけ出し直させる
retry = chat_batch([messages(reviews[i]) + [{"role": "assistant", "content": outputs[i]},
                    {"role": "user", "content": "出力が JSON として読めません。例と同じ形の JSON だけを出力し直してください。"}] for i in failed], max_new_tokens=300) if failed else []
for i, o in zip(failed, retry):
    extracted[i] = validate(o, reviews[i])
print("出し直しても読めなかった口コミ:", sum(e is None for e in extracted))

pred_df = pd.DataFrame([{"review": i, "aspect": x["観点"], "polarity": x["評価"], "evidence": str(x.get("根拠", "")).strip(), "ok": x["ok"]}
                        for i, e in enumerate(extracted) for x in (e or [])]).drop_duplicates(["review", "aspect"])
print("根拠が口コミにそのまま出てこない項目:", int((~pred_df["ok"]).sum()), "/", len(pred_df))

# 正解と比べる（実際の口コミでは、数十件に人が正解を付けて確かめる）
m = gold_df.merge(pred_df, on=["review", "aspect"], how="outer", suffixes=("_gold", "_pred"), indicator=True)
both = m[m["_merge"] == "both"]
print(f"観点の検出: 再現率 {len(both) / len(gold_df):.3f}  適合率 {len(both) / len(pred_df):.3f}  "
      f"| 検出できた観点の極性の正解率 {(both['polarity_gold'] == both['polarity_pred']).mean():.3f}")
errors = m[(m["_merge"] != "both") | (m["polarity_gold"] != m["polarity_pred"])]
print(errors["_merge"].map({"left_only": "見落とし", "right_only": "余分", "both": "極性の誤り"}).value_counts().to_dict(),
      "見落とした観点:", errors[errors["_merge"] == "left_only"].groupby("aspect")["polarity_gold"].value_counts().to_dict())

JSON が読めなかった口コミ: 18 例: ['[{"観点": "部屋の広さ", "評価": "否定", "根拠": "部屋は狭めで">}, {"観点": "清潔さ", "評価": "肯定", "根拠": "古い建物ですが、清掃はしっかりしていました."}]', '[{"観点": "朝食", "評価": "肯定", "根拠": "地元の食材を使った朝食がよかったです。"}, {"観点": "サービス（接客）", "評価": "不明", "根拠": 口コミに接客に関する言及がないため"}]\n\n※ 口コミ']
出し直しても読めなかった口コミ: 3
根拠が口コミにそのまま出てこない項目: 2 / 447
観点の検出: 再現率 0.920  適合率 1.000  | 検出できた観点の極性の正解率 1.000
{'見落とし': 39, '余分': 0, '極性の誤り': 0} 見落とした観点: {('価格', '否定'): 4, ('価格', '肯定'): 3, ('接客', '肯定'): 5, ('朝食', '否定'): 2, ('清潔さ', '肯定'): 4, ('立地', '肯定'): 11, ('立地', '否定'): 5, ('部屋の広さ', '否定'): 4, ('部屋の広さ', '肯定'): 1}


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from statsmodels.stats.proportion import proportion_confint

# 段階 2: 集計はコードで行う。割合には 95% の信頼区間（下限〜上限）を付け、代表的な意見は、根拠の文のうちグループの中心に近いものを選ぶ
def representative(texts):
    if len(texts) <= 1:
        return texts[0] if texts else "なし"
    X = TfidfVectorizer(analyzer="char", ngram_range=(2, 3)).fit_transform(texts)
    return texts[int(np.argmax(X @ np.asarray(X.mean(axis=0)).T))]

rows = []
for a in ASPECTS:
    d = pred_df[pred_df["aspect"] == a]
    pos, n = int((d["polarity"] == "肯定").sum()), len(d)
    lo, hi = proportion_confint(pos, n, method="wilson")
    ev = d[d["ok"]]      # 代表的な意見には、口コミにそのまま出てくる根拠だけを使う
    rows.append({"観点": a, "言及": n, "肯定": pos, "否定": n - pos, "肯定率": round(pos / n, 3), "下限": round(lo, 2), "上限": round(hi, 2),
                 "肯定の例": representative(ev[ev["polarity"] == "肯定"]["evidence"].tolist()),
                 "否定の例": representative(ev[ev["polarity"] == "否定"]["evidence"].tolist())})
table = pd.DataFrame(rows).set_index("観点")
print(table[["言及", "肯定", "否定", "肯定率", "下限", "上限"]])
print(table[["肯定の例", "否定の例"]].to_string())

# 段階 3: 集計表だけを LLM に渡して文章にする。数字はコードが計算したものを使わせ、件数も添える
# 多数派かどうかもコードで判定して渡す。信頼区間が 0.5 をまたぐなら「賛否が分かれる」とする
verdict = lambda r: "肯定が多数" if r["下限"] > 0.5 else "否定が多数" if r["上限"] < 0.5 else "賛否が分かれる"
digest = "\n".join(f"- {a}: 言及 {r['言及']} 件、肯定 {r['肯定']} 件 ({r['肯定率']:.0%})、否定 {r['否定']} 件 ({1 - r['肯定率']:.0%})、{verdict(r)}。"
                   f"肯定の例「{r['肯定の例']}」 否定の例「{r['否定の例']}」" for a, r in table.iterrows())
staged_prompt = (f"次は、あるビジネスホテルの口コミ {len(reviews)} 件を観点ごとに集計した結果です。\n\n{digest}\n\n"
                 "この集計だけをもとに、口コミの要約を書いてください。割合は集計の値をそのまま使ってください。"
                 "件数の少ない意見は「一部」「少数」などと書き、多数派の意見のように書かないでください。まとめでは、各観点の「肯定が多数」「否定が多数」「賛否が分かれる」の判定に従ってください。"
                 f"次の形式で、○ には数字を入れてください。\n{FORMAT}")
staged = chat(staged_prompt)
print(f"--- 段階的\n{staged}")

       言及  肯定  否定    肯定率    下限    上限
観点                                  
立地     85  79   6  0.929  0.85  0.97
部屋の広さ  91  25  66  0.275  0.19  0.37
清潔さ    63  56   7  0.889  0.79  0.95
朝食     84  60  24  0.714  0.61  0.80
接客     67  31  36  0.463  0.35  0.58
価格     57  40  17  0.702  0.57  0.80
                              肯定の例                    否定の例
観点                                                        
立地     近くにコンビニや飲食店が多く、立地は申し分ありません。  入口がわかりにくく、地図を見ても迷いました。
部屋の広さ         部屋がゆったりしていて、くつろげました。     写真で見るより部屋が小さく感じました。
清潔さ             ほこり一つなく、きれいな部屋でした。        夜中に部屋でゴキブリが出ました。
朝食             地元の食材を使った朝食がよかったです。      朝食は品数が少なく、少し残念でした。
接客          周辺のおすすめの店を親切に教えてくれました。   スタッフの言葉づかいが少し気になりました。
価格               値段のわりに設備が整っていました。      週末は料金が高く、割高に感じました。
--- 段階的
立地: 肯定 93%、否定 7%。代表的な意見としては、「近くにコンビニや飲食店が多く、立地は申し分ありません」という肯定的な意見がある一方、「入口がわかりにくく、地図を見ても迷いました」という否定的な意見も少数ありました。

部屋の広さ: 肯定 28%、否定 72%。代表的な意見としては、「部屋がゆったりしていて、くつろげました」という肯定的な意見がある一方、「写真で見るより部屋が小さく感じました」という否定的な意見がmajorityを占めて

In [6]:
# 要約の主張を検査する: 「立地については」のような観点の書き出しから次の観点までを 1 区切りとし、その中で最初に肯定か否定の語と同じ句にある割合を読み取る
MARK = r"(立地|部屋の広さ|広さ|清潔さ|朝食|接客|価格)(?:に関しては|については|は|[:：])"
PCT, POS, NEG = r"(\d+(?:\.\d+)?)\s*[%％]", "肯定|満足|好意|好評|高く評価|良い|よい", "否定|不満|苦情"

def claims(summary):
    found, marks = {}, list(re.finditer(MARK, summary))
    for k, mk in enumerate(marks):
        a = "部屋の広さ" if mk.group(1) == "広さ" else mk.group(1)
        segment = summary[mk.start():marks[k + 1].start() if k + 1 < len(marks) else len(summary)]
        for clause in re.split(r"[。、\n]", segment):     # 「肯定 ○%、否定 ○%」のように並ぶので、読点でも区切る
            if a in found:
                break
            for words, sign in [(POS, 1), (NEG, -1)]:   # 「肯定 ○%」「○% の口コミが満足」など。否定の割合なら 1 から引く
                if m := re.search(rf"(?:{words}).*?{PCT}", clause) or re.search(rf"{PCT}.*?(?:{words})", clause):
                    v = float(m.group(1)) / 100
                    found[a] = (v if sign > 0 else 1 - v, clause.strip())
                    break
    return found

def side(p):   # 6 割以上を多数派とみなす
    return "肯定が多数" if p >= 0.6 else "否定が多数" if p <= 0.4 else "賛否が分かれる"

summaries = {**one_shot, "段階的": staged}
parsed = {k: claims(s) for k, s in summaries.items()}
check = pd.DataFrame({"正解": truth["肯定率"], "段階 1 の集計": table["肯定率"],
                      **{k: pd.Series({a: v for a, (v, _) in c.items()}, dtype=float) for k, c in parsed.items()}}).reindex(list(ASPECTS))
print("要約が主張する肯定率（NaN は割合を読み取れなかった観点）")
print(check.round(2))
for k in check.columns[1:]:
    ok = check[k].notna()
    print(f"{k:12} 読み取れた観点 {ok.sum()}  正解との差の平均 {(check[k] - check['正解'])[ok].abs().mean():.3f}  "
          f"多数派の向きが合う観点 {sum(side(p) == side(t) for p, t in zip(check[k][ok], check['正解'][ok]))}")
print("\n読み取った句（一括（順序 0））:")
for a, (v, s) in parsed["一括（順序 0）"].items():
    print(f"  {a}: {v:.0%} ← {s}")

# 少数の強い苦情（清潔さの否定）に、要約がどう触れているか
complaint = re.compile(r"ゴキブリ|カビ|髪の毛|不潔|害虫")
print(f"\n清潔さの否定は正解で {1 - truth.loc['清潔さ', '肯定率']:.0%}")
for k, s in summaries.items():
    print(f"{k}: {[l.strip()[:80] for l in re.split(r'(?<=。)|\n', s) if complaint.search(l)]}")

要約が主張する肯定率（NaN は割合を読み取れなかった観点）
         正解  段階 1 の集計  一括（順序 0）  一括（順序 1）  一括（順序 2）   段階的
立地     0.89      0.93       0.6       0.6       0.6  0.93
部屋の広さ  0.27      0.28       0.3       NaN       NaN  0.28
清潔さ    0.90      0.89       NaN       0.8       0.8  0.89
朝食     0.70      0.71       0.5       0.7       0.7  0.71
接客     0.50      0.46       0.7       0.5       0.5  0.46
価格     0.67      0.70       0.4       0.3       0.3  0.70
段階 1 の集計     読み取れた観点 6  正解との差の平均 0.022  多数派の向きが合う観点 6
一括（順序 0）     読み取れた観点 5  正解との差の平均 0.198  多数派の向きが合う観点 2
一括（順序 1）     読み取れた観点 5  正解との差の平均 0.152  多数派の向きが合う観点 4
一括（順序 2）     読み取れた観点 5  正解との差の平均 0.152  多数派の向きが合う観点 4
段階的          読み取れた観点 6  正解との差の平均 0.022  多数派の向きが合う観点 6

読み取った句（一括（順序 0））:
  立地: 60% ← 約 60% の口コミが肯定的な意見を寄せており
  部屋の広さ: 30% ← 肯定的な意見が約 30%
  朝食: 50% ← 約 50% の口コミが肯定的な感想を述べていますが
  接客: 70% ← 約 70% の口コミで肯定的な対応が評価されていますが
  価格: 40% ← コストパフォーマンスが良いとする意見が約 40% ありますが

清潔さの否定は正解で 10%
一括（順序 0）: []
一括（順序 1）: []
一括（順序 2）: []
段階的: ['代表的な意見としては、「ほこり一つなく、きれいな部屋で

In [7]:
# 割合の数字が正しくても、まとめの文で多数派を言い違えることがある。「まとめ」「総じて」「全体」で始まる最初の文から最後までをまとめの部分とし、
# 観点と評価の言葉が同じ句にあれば、その観点についての主張とみなして検査する
TOPIC = {"立地": "立地", "部屋の広さ": "部屋|広さ", "清潔さ": "清潔", "朝食": "朝食", "接客": "接客|スタッフ", "価格": "価格|料金"}
STANCE = [("賛否が分かれる", "賛否|両論|分かれ|半々|ばらつき"), ("否定が多数", "否定的な意見が多|不満を持つ声が多|不満が多|改善の余地|不評"),
          ("肯定が多数", "肯定的|好評|高評価|評価が高|評価は高|満足")]

def overall_claims(summary):
    found, in_summary = [], False
    for sent in re.split(r"(?<=。)|\n", summary):
        in_summary = in_summary or bool(re.match(r"\s*(?:まとめ|総じて|全体)", sent))
        if not in_summary:
            continue
        for clause in re.split(r"おり、|ですが、|ものの、|一方、?|が、", sent):
            stance = next((label for label, pat in STANCE if re.search(pat, clause)), None)
            for a, key in TOPIC.items():   # 「価格を除けば」「価格以外は」の観点は除く
                if stance and re.search(key, clause) and not re.search(rf"(?:{key})(?:を除|以外)", clause):
                    found.append({"観点": a, "主張": stance, "正解": side(truth.loc[a, "肯定率"]), "句": clause.strip()[:30]})
    return pd.DataFrame(found, columns=["観点", "主張", "正解", "句"])

for k, s in summaries.items():
    c = overall_claims(s)
    print(f"--- {k}: 合う {int((c['主張'] == c['正解']).sum())} / 主張 {len(c)}")
    if (c["主張"] != c["正解"]).any():
        print(c[c["主張"] != c["正解"]].to_string(index=False))

# 段階的な方法では、集計の判定と食い違う主張を伝えて、1 回だけ書き直させる
# 多数派の基準は 2 つある。上の検査は、正解の肯定率と 6 割の基準（side）で採点する（要約が書く割合には件数がなく、区間を作れないため）。
# 書き直しの指示は、LLM に渡した集計の信頼区間（verdict）で判定する。実際の口コミには正解がないので、使えるのは verdict のほうである
c = overall_claims(staged)
wrong = [f"「{r.観点}」を「{r.主張}」と書いていますが、集計では「{verdict(table.loc[r.観点])}」です。" for r in c.itertuples() if r.主張 != verdict(table.loc[r.観点])]
if wrong:
    staged = chat_batch([[{"role": "user", "content": staged_prompt}, {"role": "assistant", "content": staged},
                          {"role": "user", "content": "まとめに集計と合わない記述があります。" + "".join(wrong) + "形式はそのままで、全体を書き直してください。"}]])[0]
    c = overall_claims(staged)
    print(f"\n--- 書き直した段階的な要約: 合う {int((c['主張'] == c['正解']).sum())} / 主張 {len(c)}、割合 {[f'{a} {v:.0%}' for a, (v, _) in claims(staged).items()]}")
    print(staged.splitlines()[-1])
    print(c[c["主張"] != c["正解"]].to_string(index=False))
print(f"GPU メモリの最大使用量: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

--- 一括（順序 0）: 合う 2 / 主張 6
 観点    主張      正解                              句
 接客 肯定が多数 賛否が分かれる                総じて、立地と接客は好評である
清潔さ 否定が多数   肯定が多数 部屋の広さや清潔さ、朝食の内容や価格については改善の余地があ
 朝食 否定が多数   肯定が多数 部屋の広さや清潔さ、朝食の内容や価格については改善の余地があ
 価格 否定が多数   肯定が多数 部屋の広さや清潔さ、朝食の内容や価格については改善の余地があ
--- 一括（順序 1）: 合う 4 / 主張 5
観点    主張    正解                         句
価格 否定が多数 肯定が多数 価格と部屋の広さに関しては不満を持つ声が多かった。
--- 一括（順序 2）: 合う 3 / 主張 6
   観点      主張      正解                         句
部屋の広さ 賛否が分かれる   否定が多数 部屋の広さと価格に対する満足度にはばらつきがある。
   価格 賛否が分かれる   肯定が多数 部屋の広さと価格に対する満足度にはばらつきがある。
   接客   肯定が多数 賛否が分かれる              接客は概ね好評であった。
--- 段階的: 合う 4 / 主張 6
   観点      主張    正解                          句
部屋の広さ 賛否が分かれる 否定が多数 部屋の広さと接客、価格については賛否両論があります。
   価格 賛否が分かれる 肯定が多数 部屋の広さと接客、価格については賛否両論があります。

--- 書き直した段階的な要約: 合う 5 / 主張 7、割合 ['立地 93%', '部屋の広さ 28%', '清潔さ 89%', '朝食 71%', '接客 46%', '価格 70%']
まとめ: 全体的に、立地と清潔さに関しては肯定的な意見が多数を占めており、朝食も肯定的な意見が多いですが、部屋の広さについては否定的な意見がmajorityを占めており、接客と価格については賛否両論があります。全体としては、価格を除けば概ね高評価を得ているビジネスホテルと言えます。ただ

In [8]:
from transformers import AutoModelForSequenceClassification, PreTrainedTokenizerFast

# 参考: この章の「要約が原文と食い違っていないか検査する」の NLI で、割合の主張を確かめられるか試す
nli_name = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
# transformers 5 系の DebertaV2Tokenizer は全角の記号や英数字を [UNK] にしてしまう (issue #49622) ので、tokenizer.json から直接読む
nli_tok = PreTrainedTokenizerFast.from_pretrained(nli_name)
nli = AutoModelForSequenceClassification.from_pretrained(nli_name).eval()
pairs = [("接客は肯定的な意見と否定的な意見がほぼ半々である。", "接客は肯定的な意見が多い。"),
         ("接客は肯定的な意見と否定的な意見がほぼ半々である。", "接客は賛否が分かれる。"),
         ("清潔さは肯定的な意見が大多数である。", "清潔さには不満が多い。"),
         ("清潔さは肯定的な意見が大多数である。", "清潔さについては改善の余地があると感じる宿泊客もいる。")]
with torch.no_grad():
    for premise, hypothesis in pairs:
        p = nli(**nli_tok(premise, hypothesis, return_tensors="pt")).logits.softmax(-1)[0]
        print(f"含意 {p[0]:.2f} 中立 {p[1]:.2f} 矛盾 {p[2]:.2f} | {premise} → {hypothesis}")

含意 0.98 中立 0.01 矛盾 0.01 | 接客は肯定的な意見と否定的な意見がほぼ半々である。 → 接客は肯定的な意見が多い。
含意 1.00 中立 0.00 矛盾 0.00 | 接客は肯定的な意見と否定的な意見がほぼ半々である。 → 接客は賛否が分かれる。
含意 0.00 中立 0.00 矛盾 1.00 | 清潔さは肯定的な意見が大多数である。 → 清潔さには不満が多い。
含意 0.00 中立 1.00 矛盾 0.00 | 清潔さは肯定的な意見が大多数である。 → 清潔さについては改善の余地があると感じる宿泊客もいる。


- 全部を 1 回で渡す方法は、口コミ 200 件（13,508 字）で 7,045 トークンになり、このモデルの文脈長 8,192 トークンに出力の分を残して入るぎりぎりである（300 件では 10,623 トークンで入らない）。出てきた割合は「約 60%」のような 10% 刻みの値で、数えた値とは考えにくい。並び順だけを変えた 3 回で、立地は 3 回とも 60%（正解 89%）、価格は 30〜40%（正解 67%）、接客は 70% と 50%（正解 50%）で、正解との差の平均は 0.15〜0.20 だった。順序 0 では「部屋の清潔さや広さ」と 2 つの観点をまとめて書き、清潔さの割合は読み取れなかった。
- 少数の強い苦情（清潔さの否定は正解で 10%）について、一括の 3 回のどれもが苦情の中身（ゴキブリ、カビなど）には触れなかった。そのかわり順序 0 は、清潔さを部屋の広さとまとめて「否定的な意見が約 70%」とし、まとめでは肯定が 9 割の清潔さや 7 割の朝食まで「改善の余地がある」とした。順序 1 は肯定が 7 割近い価格を「不満を持つ声が多かった」とし、順序 2 は賛否が半々の接客を「概ね好評」とした。少数意見の中身を落とすことと、全体の評価をゆがめることの両方が起きる。
- 段階的な方法の抽出は、観点の再現率 0.920、適合率 1.000、極性の正解率 1.000 で（JSON が読めなかった 18 件は、出し直しで 3 件に減った）、集計した肯定率と正解の差は平均 0.022 だった。ただし、件数は少ないが見落としは偏っていて、立地の否定は 11 件中 5 件を見落とした（肯定は 90 件中 11 件）ため、立地の肯定率は 0.89 が 0.93 になった。口コミは定型の文を組み合わせた架空のものなので、皮肉や比較、暗黙の評価を含む実際の口コミでは精度はこれより下がると考えられる。数十件に人が正解を付けて、見落としの偏りも確かめる。
- 段階的な方法の要約は、集計表の割合と代表的な意見（表の「肯定の例」「否定の例」）をそのまま写し、ゴキブリの苦情は「少数」と書いた。しかし、まとめの文では、集計で否定が多数の部屋の広さと肯定が多数の価格を「賛否両論」とし、「価格を除けば概ね高評価」と集計と逆のことも書いた（「〜を除けば」の観点は検査で除外しているので、この誤りは検出されていない）。誤りを伝えて書き直させると、部屋の広さは「否定的な意見が majority を占め」と直ったが（英単語が混ざり、検査の語に合わないので数に入っていない）、価格は直らなかった。さらに、書き足した「ただし」の文で、賛否が分かれる接客について「改善の余地があるかもしれません」と、新しい言い過ぎを加えた（合う 5 / 主張 7）。書き直させるたびに検査し直す。重要な用途では、観点ごとの行と多数派の判定はコードのテンプレートで作り、LLM には代表的な意見の言い換えだけを任せる。
- 割合には件数と信頼区間を添える。接客の肯定率 0.46 は 95% 区間が 0.35〜0.58 で 0.5 をまたぐので、「否定が多い」ではなく「賛否が分かれる」と書くのが正しい。口コミが数十件しかない店や商品では、区間はさらに広がる。
- 要約の検査は、割合の表現と評価の語を正規表現で読み取る簡単なもので、「約 40% の口コミでは部屋が狭く」のような書き方は読み取れない（表の NaN）。この章の「要約が原文と食い違っていないか検査する」の NLI は、4 組のうち 3 組（「賛否が分かれる」は含意、「不満が多い」は矛盾、「改善の余地があると感じる宿泊客もいる」は中立）は妥当に判定したが、「ほぼ半々」から「肯定的な意見が多い」を含意 0.98 と判定し、この例では割合の言い違いを見分けられなかった。割合の主張は、数えた結果と突き合わせて確かめる。
- 段階的な方法は、この章の「長い文書を分割して要約する」の map-reduce と同じ形だが、まとめる段階を LLM ではなくコードの集計にしている。部分要約を LLM にまとめさせると件数の情報が失われるので、割合を伝える要約では、数える部分をコードに任せる。
- 正解の割合がわかるように、口コミはこのレシピのために作った（テキスト分類の章の「観点ごとにポジティブ・ネガティブを判定する（アスペクトベース感情分析）」の chABSA は有価証券報告書の文なので使っていない）。実際の口コミでは、楽天トラベルのレビューや、その約 7 万文に 7 項目の観点と極性を付けたコーパスがあるが、国立情報学研究所の IDR を通じて、国内の大学と公的研究機関の研究者に学術研究の目的に限って提供されている。Amazon の多言語レビューコーパス (MARC) は配布が終了し、ライセンスも非商用の学術研究に限られていた。モデルは sarashina2.2-3b-instruct（SB Intuitions、MIT ライセンス）と mDeBERTa-v3-base-mnli-xnli（MIT ライセンス）。数値は 検証の GPU で float16 の経路（`BF16 = False`、T4 と同じ dtype）で実行したもので、GPU メモリの最大使用量は 8.9 GB だった。トークナイザーの修正については、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。